# Hyperspectral material discrimination over Mussafah — NASA EMIT

**Cool Roof Priority Index · Arab Youth Space Hackathon 2026 · Challenge 813 · Team T0050**

This notebook is the hyperspectral half of the submission. `02_main_analysis.ipynb`
establishes *how much* cooling a unit of albedo buys in Mussafah (−4.56 °C per unit
albedo) and ranks 3,603 individual roofs. This notebook asks the question multispectral
imagery cannot answer: **what are those dark roofs made of?** Bitumen can be coated.
Degraded metal has to be replaced. Those are different line items in a municipal budget.

**Instrument.** NASA EMIT (Earth Surface Mineral Dust Source Investigation), an imaging
spectrometer on the ISS: 60 m pixels, 285 bands, 381–2493 nm. EMIT's science remit is
arid-region surface mineralogy, which is precisely why 47 granules cover Mussafah —
far better coverage than the curated commercial hyperspectral archives.

**What you need.** A free NASA Earthdata account (https://urs.earthdata.nasa.gov/users/new).
The cell below prompts for it; the password is hidden as you type.

**Runtime.** 15–25 minutes, almost all of it the ~3.5 GB granule download.
Runs top-to-bottom in a fresh Google Colab runtime with no prior state.

**Honest framing up front.** The headline finding here is partly a *negative* result:
67% of the imaged scene is spectrally average, because aeolian dust homogenises surface
spectra in an arid industrial zone. That is a real and useful constraint for anyone
planning hyperspectral work in the Gulf, and it is reported rather than buried.

## 1 · Environment

In [ ]:
%pip install -q earthaccess==0.11.0 h5py netCDF4 rasterio scikit-learn \
    matplotlib pandas

In [ ]:
import os, sys, json, glob, getpass, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import h5py, earthaccess
from datetime import datetime
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)
plt.rcParams["figure.dpi"] = 110

# Outputs land next to the repo if we are inside it, otherwise in /content.
if os.path.isdir("../results") and os.path.isdir("../data"):
    BASE = ".."
elif os.path.isdir("/content"):
    BASE = "/content/coolroof_outputs"
else:
    BASE = "."
RESULTS, SAMPLE = f"{BASE}/results", f"{BASE}/data/sample_input"
for d in (RESULTS, SAMPLE):
    os.makedirs(d, exist_ok=True)
print("outputs ->", os.path.abspath(RESULTS))
print("scratch ->", os.path.abspath(SAMPLE), "(the 3.5 GB .nc goes here, NOT to git)")

## 2 · Configuration

In [ ]:
AOI_NAME = "Mussafah, Abu Dhabi"
AOI_BBOX = [54.43, 24.29, 54.58, 24.41]        # W, S, E, N — same AOI as 02_main

# Carried over from 02_main_analysis.ipynb so the two notebooks sit on one scale.
SENS   = 4.56      # °C of cooling per unit of albedo increase, measured locally
TARGET = 0.60      # albedo a commercial cool coating reaches

# Pinned so a reviewer analyses the SAME scene we did. If this granule ever leaves
# the archive the scoring block below picks the next best one and says so.
PINNED_GRANULE = "EMIT_L2A_RFL_001_20260520T110418_2614007_042"

K = 5              # material classes
HALFWIDTH = 25     # nm, half-width of the band-depth shoulder/centre windows

# Diagnostic absorption features for urban surfaces.
#   name:            (left_nm, centre_nm, right_nm, what it diagnoses)
FEATURES = {
    "fe_oxide_900":   (760,  900,  1050, "Fe-oxide — rust, red tile, ferruginous sand"),
    "bitumen_1730":   (1660, 1730, 1780, "C-H stretch — BITUMEN / ASPHALT"),
    "alOH_2200":      (2120, 2200, 2255, "Al-OH — clay, cement, concrete"),
    "carbonate_2340": (2270, 2340, 2420, "carbonate — limestone, sabkha sand"),
}

# Sentinel-2 passband centres/widths and Bonafoni & Sekertekin (2020) weights.
# Reused verbatim from 02_main_analysis so EMIT albedo is directly comparable.
S2W = {"B02": (490, 65), "B03": (560, 35), "B04": (665, 30),
       "B08": (842, 115), "B11": (1610, 90), "B12": (2190, 180)}
AW  = dict(B02=0.2266, B03=0.1236, B04=0.1573, B08=0.3417, B11=0.1170, B12=0.0338)
assert abs(sum(AW.values()) - 1.0) < 1e-9, "albedo weights must sum to 1"
print(f"{AOI_NAME}  bbox={AOI_BBOX}")
print(f"sensitivity {SENS} °C per unit albedo (from 02_main_analysis), target α={TARGET}")

## 3 · Authenticate with NASA Earthdata

EMIT is free and open, but the LP DAAC requires a (free) account. Register at
https://urs.earthdata.nasa.gov/users/new — takes about a minute. The password
prompt below is hidden as you type and nothing is written to the notebook.

In [ ]:
print("NASA Earthdata login (password is hidden as you type):")
os.environ["EARTHDATA_USERNAME"] = input("  username: ").strip()
os.environ["EARTHDATA_PASSWORD"] = getpass.getpass("  password: ")
auth = earthaccess.login(strategy="environment", persist=True)
print("authenticated:", bool(auth))
if not auth:
    raise RuntimeError("Earthdata login failed — check the username/password and re-run.")

## 4 · Find and select a granule

47 EMIT L2A granules intersect the AOI. Which one you pick matters a great deal, so
the criteria are explicit and scored rather than left to chance:

| criterion | why |
|---|---|
| **summer (May–Sept)** | matches the Landsat epoch-B summer composite in `02_main_analysis`, so the material map describes the same surfaces as the thermal analysis |
| **recent (2024+)** | industrial roofs get re-covered; a 2022 scene describes roofs that may no longer exist |
| **sun 10:00–15:30 local** (06:00–11:30 UTC) | low sun throws long shadows between industrial blocks, and a shadowed pixel's spectrum is the shadow's, not the roof's |
| **largest AOI overlap** | EMIT swaths are narrow and most granules clip only a corner of Mussafah |

The selected granule is pinned by ID so a reviewer gets the identical scene. The
scoring runs anyway and prints the top candidates, so the choice is auditable and
degrades gracefully if the pinned ID ever disappears.

In [ ]:
print("Searching EMIT L2A (EMITL2ARFL) over the AOI...")
granules = earthaccess.search_data(short_name="EMITL2ARFL",
                                   bounding_box=tuple(AOI_BBOX), count=200)
print(f"{len(granules)} granules intersect the AOI")

def gmeta(g):
    """Pull date, name and crude AOI-overlap fraction out of a CMR UMM record."""
    try:
        u = g["umm"]
        t  = u["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
        dt = datetime.fromisoformat(t.replace("Z", "+00:00"))
        poly = (u["SpatialExtent"]["HorizontalSpatialDomain"]["Geometry"]
                 ["GPolygons"][0]["Boundary"]["Points"])
        lons = [p["Longitude"] for p in poly]; lats = [p["Latitude"] for p in poly]
        ow = max(0, min(max(lons), AOI_BBOX[2]) - max(min(lons), AOI_BBOX[0]))
        oh = max(0, min(max(lats), AOI_BBOX[3]) - max(min(lats), AOI_BBOX[1]))
        frac = (ow * oh) / ((AOI_BBOX[2]-AOI_BBOX[0]) * (AOI_BBOX[3]-AOI_BBOX[1]))
        return dict(g=g, dt=dt, name=u.get("GranuleUR", ""), overlap=frac,
                    utc_hour=dt.hour + dt.minute/60)
    except Exception:
        return None

cands = [m for m in (gmeta(g) for g in granules) if m]
for m in cands:
    m["summer"]   = 5 <= m["dt"].month <= 9
    m["good_sun"] = 6.0 <= m["utc_hour"] <= 11.5          # local 10:00–15:30
    m["score"] = (m["overlap"] * 3
                  + (2 if m["summer"] else 0)
                  + (2 if m["good_sun"] else 0)
                  + (m["dt"].year - 2022) * 0.4)
cands.sort(key=lambda m: -m["score"])

print(f"\n{'granule':48s} {'date':12s} {'local':7s} {'ovl':>5s} {'sum':>4s} {'sun':>4s} {'score':>6s}")
print("-" * 95)
for m in cands[:12]:
    loc = (m["utc_hour"] + 4) % 24
    print(f"{m['name'][:48]:48s} {m['dt'].date()!s:12s} "
          f"{int(loc):02d}:{int((loc%1)*60):02d}   {m['overlap']:5.2f} "
          f"{'Y' if m['summer'] else '-':>4s} {'Y' if m['good_sun'] else '-':>4s} "
          f"{m['score']:6.2f}")

pinned = [m for m in cands if m["name"] == PINNED_GRANULE]
if pinned:
    best = pinned[0]
    print(f"\nUsing the PINNED granule (rank {cands.index(best)+1} by score)")
else:
    best = cands[0]
    print(f"\n!! Pinned granule {PINNED_GRANULE} not found in the archive.")
    print("   Falling back to the top-scoring candidate. Numbers below will differ")
    print("   from the README, which reports the pinned scene.")

print(f"\nSELECTED: {best['name']}")
print(f"  acquired  {best['dt']} UTC   (local {(best['utc_hour']+4)%24:.1f} h)")
print(f"  AOI overlap {best['overlap']*100:.0f}%   summer={best['summer']}   "
      f"good_sun={best['good_sun']}")

## 5 · Download

~3.5 GB, usually 5–15 minutes on a Colab runtime. The `.nc` goes to
`data/sample_input/` and **must never be committed** — the repo `.gitignore`
excludes `*.nc` for exactly this reason.

In [ ]:
print("Downloading (~3.5 GB)...")
got = earthaccess.download([best["g"]], SAMPLE)
nc = [f for f in got if str(f).endswith(".nc")]
if not nc:
    raise RuntimeError(f"no .nc in the download: {got}")
NC = str(nc[0])
print(f"file: {NC}")
print(f"size: {os.path.getsize(NC)/1e9:.2f} GB")

## 6 · Orthorectify the AOI window

EMIT ships in **sensor geometry** — the array is indexed by detector row and
cross-track element, not by latitude and longitude — plus a **Geometric Lookup Table
(GLT)** that maps each cell of a regular geographic grid back to the sensor pixel that
fell there. Orthorectification is therefore a gather, not a resample.

Rather than build the whole ortho grid and then crop, this builds only the AOI window
of the grid, reads the single rectangular slab of the reflectance cube that window
needs, and gathers. Reading the full ~1.8 GB array would also work; it is simply
wasteful, and on a free Colab runtime it is the difference between working and OOM.

Two details that matter:
- The GLT is **1-based**, with 0 meaning "no sensor pixel maps here". Off-by-one here
  silently shifts the whole image by one pixel.
- EMIT's fill value is −9999. Left in, it dominates every mean.

In [ ]:
print("Orthorectifying the AOI window...")
with h5py.File(NC, "r") as f:
    wl    = f["sensor_band_parameters/wavelengths"][:]
    fwhm  = f["sensor_band_parameters/fwhm"][:]
    good  = f["sensor_band_parameters/good_wavelengths"][:].astype(bool)
    glt_x = f["location/glt_x"][:].astype(np.int32)
    glt_y = f["location/glt_y"][:].astype(np.int32)
    gt    = np.array(f.attrs["geotransform"]).ravel()

    ny_o, nx_o = glt_x.shape
    lon = gt[0] + gt[1] * (np.arange(nx_o) + 0.5)
    lat = gt[3] + gt[5] * (np.arange(ny_o) + 0.5)
    print(f"  ortho grid {ny_o} x {nx_o}, pixel {abs(gt[1])*111320:.0f} m")
    print(f"  granule extent  lon {lon.min():.3f}..{lon.max():.3f}   "
          f"lat {lat.min():.3f}..{lat.max():.3f}")
    print(f"  AOI             lon {AOI_BBOX[0]:.3f}..{AOI_BBOX[2]:.3f}   "
          f"lat {AOI_BBOX[1]:.3f}..{AOI_BBOX[3]:.3f}")

    csel = np.where((lon >= AOI_BBOX[0]) & (lon <= AOI_BBOX[2]))[0]
    rsel = np.where((lat >= AOI_BBOX[1]) & (lat <= AOI_BBOX[3]))[0]
    if len(csel) < 5 or len(rsel) < 5:
        raise RuntimeError("granule does not actually cover the AOI — "
                           "set PINNED_GRANULE to the next candidate and re-run")
    r0, r1, c0, c1 = rsel.min(), rsel.max()+1, csel.min(), csel.max()+1
    print(f"  AOI window {r1-r0} x {c1-c0} px  "
          f"({(r1-r0)*abs(gt[5])*111320/1000:.1f} x "
          f"{(c1-c0)*abs(gt[1])*111320/1000:.1f} km)")

    gx, gy = glt_x[r0:r1, c0:c1], glt_y[r0:r1, c0:c1]
    filled = (gx > 0) & (gy > 0)                   # 0 = no sensor pixel maps here
    print(f"  GLT-filled: {filled.sum():,} of {filled.size:,} "
          f"({filled.mean()*100:.1f}%)")
    if filled.sum() < 500:
        raise RuntimeError("almost no valid GLT over the AOI — try another granule")

    sy, sx = gy[filled] - 1, gx[filled] - 1        # GLT is 1-BASED
    y0, y1 = int(sy.min()), int(sy.max()) + 1
    x0, x1 = int(sx.min()), int(sx.max()) + 1
    print(f"  reading sensor slab [{y0}:{y1}, {x0}:{x1}, {good.sum()} good bands]")
    slab = f["reflectance"][y0:y1, x0:x1, :][:, :, good].astype("float32")

wls = wl[good]
cube = np.full((r1-r0, c1-c0, wls.size), np.nan, dtype="float32")
cube[filled] = slab[sy - y0, sx - x0, :]
del slab
cube[cube <= -9990] = np.nan                       # EMIT fill value
cube = np.clip(cube, 0, None)                      # reflectance cannot be negative
aoi_lon, aoi_lat = lon[c0:c1], lat[r0:r1]

valid = np.isfinite(cube).sum(axis=2) > wls.size * 0.9
print(f"\ncube {cube.shape} — {wls.size} good bands, "
      f"{wls.min():.0f}–{wls.max():.0f} nm")
print(f"valid pixels: {valid.sum():,} of {valid.size:,} ({valid.mean()*100:.1f}%)")
print(f"  (the granule clips the NORTH of the AOI — see the coverage caveat in §11)")

## 7 · Drop the atmospheric water bands

EMIT's own `good_wavelengths` flag **retains** the 1340–1480 nm and 1790–1980 nm
water-vapour windows. In those windows atmospheric transmission collapses, and since
reflectance is computed as radiance ÷ irradiance, dividing by a near-zero number
produces values that reach 0.8 and are pure noise.

This does not affect any of the band depths below — all four features sit outside
those windows — but it must not appear in a figure a reviewer reads as surface
reflectance. The blue edge (<420 nm) and the far SWIR edge (>2460 nm) are dropped for
the same reason: low SNR.

In [ ]:
def atmospheric(w):
    return (1340 <= w <= 1480) or (1790 <= w <= 1980) or w > 2460 or w < 420

keep = np.array([not atmospheric(w) for w in wls])
wlc   = wls[keep]
cubec = cube[:, :, keep]
ny, nx, nb = cubec.shape

print(f"dropped {(~keep).sum()} bands, retained {keep.sum()}")
print(f"  windows removed: 1340–1480 nm (H2O), 1790–1980 nm (H2O), >2460 nm, <420 nm")
print(f"  retained range : {wlc.min():.0f}–{wlc.max():.0f} nm")

flat = cubec.reshape(-1, nb)
ok   = np.isfinite(flat).all(axis=1)
X    = flat[ok]
print(f"\ncomplete spectra: {X.shape[0]:,} x {nb} bands")
print(f"max reflectance now {X.max():.3f}  "
      f"(~0.8 with the water-vapour spike left in)")

## 8 · Band depths against a local two-point continuum

**Why not cluster the raw spectra?** We tried. k-means on unit-normalised EMIT spectra
returned six classes with near-identical *shapes* differing only in brightness — the
2200 nm ratio spanned 0.936–0.971 across all six. That is a brightness histogram with
extra steps, and the main analysis already measures brightness. In a dust-dominated
arid city, surface materials converge spectrally and compositional signal has to be
*extracted*.

The standard extraction is **band depth against a local two-point continuum**
(Clark & Roush 1984): draw a straight line between the mean reflectance in the
feature's left and right shoulders, and measure how far the band floor falls below it.

$$\text{depth} = 1 - \frac{R_{\text{centre}}}{R_{\text{continuum}}(\lambda_{\text{centre}})}$$

This is both the conventional definition and trivially vectorisable — it replaced a
full-spectrum convex-hull continuum removal that took ~7 minutes with one that takes
under a second, verified against synthetic spectra with known band depths.

Why these four features:

| feature | centre | diagnoses | visible to Sentinel-2? |
|---|---|---|---|
| Fe-oxide | 900 nm | rust, red tile, ferruginous sand | partly (broad) |
| **C–H stretch** | **1730 nm** | **bitumen, asphalt, membrane roofing** | **no** |
| Al-OH | 2200 nm | clay, cement, concrete | no |
| carbonate | 2340 nm | limestone aggregate, sabkha sand | no |

The 1730 nm C–H feature is the one that earns the hyperspectral data. There is no
Sentinel-2 band there at all, and it is what separates a bituminous roof (coat it)
from a dark metal roof (replace it).

In [ ]:
def band_depth(Xa, left_nm, centre_nm, right_nm, hw=HALFWIDTH):
    """1 - R_centre / R_continuum(centre); continuum = line through shoulder means."""
    L = (wlc >= left_nm   - hw) & (wlc <= left_nm   + hw)
    Cm = (wlc >= centre_nm - hw) & (wlc <= centre_nm + hw)
    R = (wlc >= right_nm  - hw) & (wlc <= right_nm  + hw)
    if not (L.any() and Cm.any() and R.any()):
        return None
    wl_l, wl_r = wlc[L].mean(), wlc[R].mean()
    r_l, r_r = Xa[:, L].mean(axis=1), Xa[:, R].mean(axis=1)
    cont = r_l + ((centre_nm - wl_l) / (wl_r - wl_l)) * (r_r - r_l)
    r_c = Xa[:, Cm].min(axis=1)                  # deepest point inside the feature
    return 1.0 - r_c / (cont + 1e-9)

depths, names = [], []
for k, (l, c, r, desc) in FEATURES.items():
    d = band_depth(X, l, c, r)
    if d is None:
        print(f"  {k:16s} shoulders fall outside the retained bands — skipped")
        continue
    depths.append(d); names.append(k)
D = np.column_stack(depths)
mu, sd = D.mean(0), D.std(0)

# Broadband albedo on exactly the scale 02_main_analysis uses, so a class's
# mean albedo can be read straight off the same axis as the roof rankings.
alb = np.zeros(X.shape[0])
for b, (c, w) in S2W.items():
    s = (wlc >= c - w/2) & (wlc <= c + w/2)
    if s.any():
        alb += AW[b] * X[:, s].mean(axis=1)
    else:
        print(f"  !! no EMIT bands inside S2 {b} — albedo is incomplete")

print("Band depths over the scene (local two-point continuum, Clark & Roush 1984):\n")
print(f"{'feature':17s} {'mean':>9s} {'sd':>8s} {'p5':>9s} {'p95':>9s}  "
      f"{'corr w/ albedo':>14s}")
print("-" * 76)
for i, n in enumerate(names):
    r = np.corrcoef(D[:, i], alb)[0, 1]
    print(f"{n:17s} {mu[i]:+9.4f} {sd[i]:8.4f} "
          f"{np.percentile(D[:,i],5):+9.4f} {np.percentile(D[:,i],95):+9.4f}  "
          f"{r:+14.3f}")

print(f"\nalbedo:  p5 {np.percentile(alb,5):.3f}   median {np.median(alb):.3f}   "
      f"p95 {np.percentile(alb,95):.3f}")
print(f"\nNote the carbonate depth: {mu[names.index('carbonate_2340')]:.4f}, an order of")
print("magnitude above every other feature. Abu Dhabi sits on carbonate sabkha, so")
print("carbonate absorption saturates the whole scene. Classifying on ABSOLUTE depth")
print("therefore labels everything 'limestone' — which is why §9 standardises first.")

## 9 · Cluster on composition, interpret by deviation from the scene mean

Two deliberate choices:

**Cluster the standardised depths, not the raw ones.** Carbonate dominates absolutely
everywhere, so raw depths cluster on "how much carbonate", which is "where is the
sand". Standardising to the scene mean makes the question *how does this pixel differ
from typical Mussafah ground*.

**Interpret by z-score.** An earlier pass labelled all five classes "limestone" for the
same reason. What identifies a surface is its **deviation** from the background: a class
enriched +1.2σ in the C–H feature is bituminous *relative to this scene*, which is the
only claim the data support.

**The independence test.** If composition clustering merely recovered brightness, the
hyperspectral data would add nothing over Sentinel-2. So the composition labels are
compared against labels from k-means on albedo alone, using the **Adjusted Rand Index**.
ARI ≈ 0 means the two partitions are independent; ARI ≈ 1 means composition *is*
brightness. This is the test that decides whether this notebook justifies itself, and
it is reported whichever way it falls.

In [ ]:
Dz = (D - mu) / (sd + 1e-9)
sub = Dz[np.random.choice(len(Dz), min(40000, len(Dz)), replace=False)]
lab = KMeans(K, n_init=20, random_state=42).fit(sub).predict(Dz)

lab_bright = KMeans(K, n_init=10, random_state=42).fit_predict(alb.reshape(-1, 1))
ari = adjusted_rand_score(lab, lab_bright)

IDENT = {
    "bitumen_1730":   "BITUMINOUS / CEMENT built surface",
    "alOH_2200":      "CEMENT / CLAY-BEARING",
    "carbonate_2340": "CARBONATE (sabkha sand, limestone aggregate)",
    "fe_oxide_900":   "IRON-BEARING (rust, red tile, ferruginous sand)",
}

rows = []
for k in range(K):
    m = lab == k
    z = {n: float(Dz[m, i].mean()) for i, n in enumerate(names)}
    enr = {n: v for n, v in z.items() if v >  0.5}
    dep = {n: v for n, v in z.items() if v < -0.5}
    top = max(z, key=z.get)
    rows.append(dict(
        material_class=k,
        pixel_share_pct=round(m.mean()*100, 1),
        mean_albedo=round(float(alb[m].mean()), 3),
        albedo_sd=round(float(alb[m].std()), 3),
        dT_if_coated_c=round(SENS * max(0, TARGET - alb[m].mean()), 2),
        **{f"z_{n}": round(z[n], 2) for n in names},
        enriched_in=", ".join(f"{n.split('_')[0]}+{z[n]:.1f}s" for n in
                              sorted(enr, key=enr.get, reverse=True)) or "—",
        depleted_in=", ".join(f"{n.split('_')[0]}{z[n]:.1f}s" for n in
                              sorted(dep, key=dep.get)) or "—",
        interpretation=(IDENT[top] if z[top] > 0.5
                        else "spectrally average / dust-coated"),
    ))
mat = pd.DataFrame(rows).sort_values("mean_albedo").reset_index(drop=True)

show = (["material_class", "pixel_share_pct", "mean_albedo", "albedo_sd",
         "dT_if_coated_c"] + [f"z_{n}" for n in names] + ["interpretation"])
print("EMIT MATERIAL CLASSES — MUSSAFAH   (z = standard deviations from the scene mean)")
print("=" * 124)
print(mat[show].to_string(index=False))
print("=" * 124)

for _, r in mat.iterrows():
    print(f"\nclass {int(r.material_class)} | {r.pixel_share_pct:4.1f}% of scene | "
          f"albedo {r.mean_albedo:.3f} (sd {r.albedo_sd:.3f}) | "
          f"coating to {TARGET} buys {r.dT_if_coated_c:.2f} °C")
    print(f"   enriched: {r.enriched_in}")
    print(f"   depleted: {r.depleted_in}")
    print(f"   -> {r.interpretation}")

dusty = mat.loc[mat.interpretation.str.startswith("spectrally average"),
                "pixel_share_pct"].sum()
print(f"\n{dusty:.0f}% of the imaged scene is spectrally average — the dust-"
      f"homogenisation result.")

In [ ]:
# -------- the headline, stated precisely enough to survive a reviewer -------
BI = "bitumen_1730"
hero = mat.loc[mat[f"z_{BI}"].idxmax()]
alb_rank = int((mat.mean_albedo < hero.mean_albedo).sum()) + 1   # 1 = darkest
ordinal = {1: "darkest", 2: "second-darkest", 3: "third-darkest"}.get(
    alb_rank, f"{alb_rank}th-darkest")
widest = hero.albedo_sd >= mat.albedo_sd.max() - 1e-9

print("=" * 78)
print("THE HYPERSPECTRAL RESULT")
print("=" * 78)
print(f"Class {int(hero.material_class)} covers {hero.pixel_share_pct}% of the imaged scene at mean")
print(f"albedo {hero.mean_albedo:.3f} — the {ordinal} of the {K} classes — and is enriched")
print(f"{hero[f'z_{BI}']:+.2f}s in the 1730 nm C-H (bitumen) feature and "
      f"{hero['z_alOH_2200']:+.2f}s in")
print(f"2200 nm Al-OH (cement).")
print()
print(f"Its albedo spread (sd {hero.albedo_sd:.3f}) is "
      f"{'the widest of any class' if widest else 'wide'}, against "
      f"{mat.albedo_sd.min():.3f}–{mat.albedo_sd.max():.3f}")
print("across the set — consistent with mixed built materials rather than uniform sand.")
print()
print("Broadband albedo tells you these surfaces are dark. Only the narrow bands tell")
print("you they are bituminous and cement-bearing — and that is what decides whether a")
print("roof can be COATED or has to be RE-COVERED, which are different line items on a")
print("municipal schedule of rates by roughly an order of magnitude.")
print()
print(f"ARI against brightness-only clustering = {ari:.3f}")
if ari < 0.3:
    print("  -> LOW. The composition partition is largely INDEPENDENT of brightness:")
    print("     the absorption features separate surfaces albedo alone cannot. This is")
    print("     the hyperspectral payoff, and it is the claim to defend on a slide.")
else:
    print("  -> HIGH. Composition largely tracks brightness in this scene. Report it as")
    print("     a finding: in a dust-dominated arid industrial zone aeolian deposition")
    print("     homogenises surface spectra, and hyperspectral adds limited material")
    print("     discrimination beyond albedo. An honest negative, and directly useful")
    print("     to the 813 mission team, who will hit the same wall.")
print("=" * 78)

## 10 · Figure

In [ ]:
def unflatten(v):
    a = np.full(flat.shape[0], np.nan); a[ok] = v
    return a.reshape(ny, nx)

cls  = unflatten(lab.astype(float))
albi = unflatten(alb)
bidx = names.index(BI)
bimg = unflatten(D[:, bidx])
ext  = [aoi_lon[0], aoi_lon[-1], aoi_lat[-1], aoi_lat[0]]

# A lon/lat image needs aspect = 1/cos(lat) to be geographically correct: at this
# latitude one degree of longitude is 101.4 km against 110.6 km for one degree of
# latitude. aspect="auto" stretches the image to whatever shape the subplot is,
# which silently misrepresents every shape on the map.
lat0   = float(np.mean(aoi_lat))
ASPECT = 1.0 / np.cos(np.radians(lat0))
km_w   = (aoi_lon[-1]-aoi_lon[0]) * 111.320 * np.cos(np.radians(lat0))
km_h   = (aoi_lat[0]-aoi_lat[-1]) * 110.574
print(f"AOI window {km_w:.1f} x {km_h:.1f} km (true aspect {km_w/km_h:.2f}); "
      f"maps drawn at aspect={ASPECT:.4f}")

fig = plt.figure(figsize=(18, 10.5))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.05])

ax = fig.add_subplot(gs[0, 0])
im = ax.imshow(albi, cmap="gray", vmin=0.12, vmax=0.42, extent=ext, aspect=ASPECT)
ax.set_title("Broadband albedo\n(what Sentinel-2 also sees)")
ax.axis("off"); plt.colorbar(im, ax=ax, fraction=0.046)

ax = fig.add_subplot(gs[0, 1])
im = ax.imshow(bimg, cmap="inferno", vmin=np.nanpercentile(bimg, 2),
               vmax=np.nanpercentile(bimg, 98), extent=ext, aspect=ASPECT)
ax.set_title("1730 nm C–H band depth\n(bitumen — no Sentinel-2 band exists here)")
ax.axis("off"); plt.colorbar(im, ax=ax, fraction=0.046)

ax = fig.add_subplot(gs[0, 2])
im = ax.imshow(cls, cmap="tab10", vmin=0, vmax=K-1, extent=ext, aspect=ASPECT)
ax.set_title(f"Material classes\nARI vs brightness-only = {ari:.2f}")
ax.axis("off"); plt.colorbar(im, ax=ax, ticks=range(K), fraction=0.046)

ax = fig.add_subplot(gs[1, :2])
for k in mat.material_class.astype(int):
    r = mat.loc[mat.material_class == k].iloc[0]
    lw = 2.6 if k == int(hero.material_class) else 1.3
    ax.plot(wlc, X[lab == k].mean(0), lw=lw,
            label=f"c{k}  α={r.mean_albedo:.3f}  ({r.pixel_share_pct:.1f}%)  "
                  f"{r.interpretation[:34]}")
for n in names:
    c = FEATURES[n][1]
    ax.axvspan(c - HALFWIDTH, c + HALFWIDTH, color="k", alpha=0.08)
    ax.text(c, ax.get_ylim()[1]*0.98, n.split("_")[0], ha="center", fontsize=7)
ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Surface reflectance")
ax.set_title("Mean spectrum per class — atmospheric water bands removed; "
             "shaded = diagnostic features; bold = the bituminous class")
ax.legend(fontsize=7.5, loc="lower right"); ax.grid(alpha=0.3)

ax = fig.add_subplot(gs[1, 2])
mz = mat.set_index("material_class")[[f"z_{n}" for n in names]]
im = ax.imshow(mz.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.split("_")[0] for n in names], rotation=35, ha="right", fontsize=8)
ax.set_yticks(range(len(mz)))
ax.set_yticklabels([f"c{i}" for i in mz.index], fontsize=8)
for i in range(len(mz)):
    for j in range(len(names)):
        v = mz.values[i, j]
        ax.text(j, i, f"{v:+.1f}", ha="center", va="center", fontsize=8,
                color="white" if abs(v) > 1.2 else "black")
ax.set_title("Composition fingerprint\n(σ from scene mean)")
plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig(f"{RESULTS}/10_emit_final.png", bbox_inches="tight", dpi=135)
plt.show()

## 11 · Export, provenance and limitations

Everything written here is small enough to commit. The 3.5 GB `.nc` is not, and
`.gitignore` excludes it — the notebook re-downloads it from the pinned granule ID.

The limitations below are not boilerplate. Each one is a thing a reviewer could
otherwise find and we would rather name first.

In [ ]:
import rasterio
from rasterio.transform import from_origin

px = abs(aoi_lon[1] - aoi_lon[0]); py = abs(aoi_lat[1] - aoi_lat[0])
tr = from_origin(aoi_lon[0] - px/2, aoi_lat[0] + py/2, px, py)
for arr, nm in [(cls, "emit_material_classes"), (albi, "emit_albedo"),
                (bimg, "emit_depth_bitumen_1730")]:
    with rasterio.open(f"{RESULTS}/{nm}.tif", "w", driver="GTiff",
                       height=arr.shape[0], width=arr.shape[1], count=1,
                       dtype="float32", crs="EPSG:4326", transform=tr,
                       nodata=np.nan, compress="deflate") as dst:
        dst.write(arr.astype("float32"), 1)

mat.to_csv(f"{RESULTS}/emit_material_classes.csv", index=False)
np.savez_compressed(f"{RESULTS}/emit_class_spectra.npz", wavelengths=wlc,
                    **{f"class_{k}": X[lab == k].mean(0) for k in range(K)})

out = {
 "source": "NASA EMIT L2A surface reflectance (EMITL2ARFL)",
 "granule": best["name"],
 "granule_pinned": bool(pinned),
 "acquired_utc": str(best["dt"]),
 "acquired_local_hour": round((best["utc_hour"] + 4) % 24, 1),
 "aoi": AOI_NAME, "aoi_bbox": AOI_BBOX, "real_aoi": True,
 "resolution_m": round(abs(gt[1]) * 111320),
 "bands_total": int(wl.size), "bands_good_flagged": int(good.sum()),
 "bands_retained": int(nb),
 "wavelength_range_nm": [int(wlc.min()), int(wlc.max())],
 "bands_removed": ("atmospheric water 1340-1480 and 1790-1980 nm, SWIR edge >2460 nm, "
                   "blue edge <420 nm — EMIT's good_wavelengths flag retains these and "
                   "reflectance there is noise, not surface signal"),
 "aoi_pixels": int(cls.size), "valid_pixels": int(valid.sum()),
 "spectra_classified": int(X.shape[0]),
 "glt_fill_pct": round(float(filled.mean()*100), 1),
 "method": ("band depths against a local two-point continuum (Clark & Roush 1984); "
            "k-means (k=5, seed 42) on depths standardised to the scene mean; classes "
            "interpreted by DEVIATION from the scene mean, because absolute carbonate "
            "depth saturates a carbonate-sabkha setting"),
 "albedo_method": ("EMIT channels averaged over Sentinel-2 passbands, then Bonafoni & "
                   "Sekertekin (2020) narrow-to-broadband weights — identical formula "
                   "to 02_main_analysis, so classes sit on the same albedo scale as "
                   "the roof rankings"),
 "sensitivity_c_per_unit_albedo": SENS,
 "ari_vs_brightness_clustering": round(float(ari), 3),
 "features": {n: {"left_nm": FEATURES[n][0], "centre_nm": FEATURES[n][1],
                  "right_nm": FEATURES[n][2], "diagnostic_of": FEATURES[n][3],
                  "scene_mean_depth": round(float(mu[i]), 4),
                  "scene_sd": round(float(sd[i]), 4),
                  "corr_with_albedo": round(float(np.corrcoef(D[:, i], alb)[0, 1]), 3)}
              for i, n in enumerate(names)},
 "headline": {
    "class": int(hero.material_class),
    "albedo_rank": alb_rank, "albedo_rank_label": ordinal,
    "pixel_share_pct": float(hero.pixel_share_pct),
    "mean_albedo": float(hero.mean_albedo),
    "albedo_sd": float(hero.albedo_sd),
    "widest_albedo_spread": bool(widest),
    "z_bitumen_1730": float(hero[f"z_{BI}"]),
    "z_alOH_2200": float(hero["z_alOH_2200"]),
    "statement": (
        "Class {c} covers {s:.1f}% of the imaged scene at mean albedo {a:.3f} — the "
        "{o} of the {k} classes — and is enriched {zb:+.2f} sigma in the 1730 nm C-H "
        "(bitumen) feature and {za:+.2f} sigma in 2200 nm Al-OH (cement), with the "
        "widest albedo spread of any class (sd {sdv:.3f}), consistent with mixed built "
        "materials rather than uniform sand. Broadband albedo identifies these surfaces "
        "as dark; only the narrow bands identify them as bituminous and cement-bearing, "
        "which is what determines whether a roof can be coated or must be replaced."
    ).format(c=int(hero.material_class), s=hero.pixel_share_pct, a=hero.mean_albedo,
             o=ordinal, k=K, zb=hero[f"z_{BI}"], za=hero["z_alOH_2200"],
             sdv=hero.albedo_sd),
 },
 "spectrally_average_share_pct": float(round(dusty, 1)),
 "classes": mat.to_dict(orient="records"),
 "why_not_raw_spectra": ("k-means on unit-normalised EMIT spectra returned six classes "
    "with near-identical shapes differing only in brightness (2200 nm ratio 0.936-0.971 "
    "across all six). In a dust-dominated arid city surface materials converge "
    "spectrally, so band-depth extraction is required to recover compositional signal."),
 "caveats": [
   "Spectral inference only. No field verification was possible: classes are "
   "'consistent with' a material, never confirmed as one. Confirming them needs a "
   "rooftop sampling campaign, which is step 1 of the proposed next phase.",
   "EMIT is 60 m, one pixel = 3,600 m2. Material class is reliable only for roofs "
   "spanning at least one clean pixel; the top-100 ranked roofs average 1.07 ha "
   "(~3 px), while the smaller roofs in the ranking are spectrally mixed.",
   "This granule clips the NORTHERN part of the AOI (granule south edge 24.353 N vs "
   "AOI south 24.29 N) at {:.1f}% GLT fill — roughly 46 km2 of the 195 km2 AOI. The "
   "material map therefore covers a subset of the ranked roofs.".format(
       float(filled.mean()*100)),
   "Both the 2200 nm Al-OH and the 1730 nm C-H depth maps show faint diagonal detector "
   "striping, and mean Al-OH depth is slightly negative scene-wide, so part of that "
   "signal is instrumental rather than surface. The class-mean z-scores average over "
   "thousands of pixels and are robust to it; a per-pixel material call on a single "
   "roof is not, and should not be made from this product alone.",
   "Carbonate absorption dominates absolutely everywhere (scene mean depth "
   "{:.3f} against {:.3f}-{:.3f} for the other features) because Abu Dhabi sits on "
   "carbonate sabkha. The classification therefore keys on deviations from that "
   "background, not on the background itself.".format(
       float(mu[names.index('carbonate_2340')]),
       float(min(mu[i] for i, n in enumerate(names) if n != 'carbonate_2340')),
       float(max(mu[i] for i, n in enumerate(names) if n != 'carbonate_2340'))),
   "Albedo is synthesised from reflectance over Sentinel-2 passbands, not measured "
   "in situ with a pyranometer.",
 ],
 "licence": "NASA EMIT data are open and free (NASA LP DAAC).",
 "citation": ("Green, R.O. et al., EMIT L2A Estimated Surface Reflectance, NASA LP DAAC. "
              "Continuum-removal method: Clark, R.N. & Roush, T.L. (1984), JGR 89(B7). "
              "Albedo weights: Bonafoni, S. & Sekertekin, A. (2020), IEEE GRSL 17(9)."),
}
with open(f"{RESULTS}/emit_final_summary.json", "w") as f:
    json.dump(out, f, indent=2)

print(f"Written to {os.path.abspath(RESULTS)}:")
for fn in sorted(os.listdir(RESULTS)):
    if fn.startswith(("emit_", "10_emit")):
        print(f"   {fn:38s} {os.path.getsize(os.path.join(RESULTS, fn))/1e3:9.1f} KB")

In [ ]:
# Zip the EMIT outputs and push them to your Downloads folder, so a runtime
# disconnect does not cost you the run.
import shutil
dst = "/content/emit_final" if os.path.isdir("/content") else "./emit_final"
shutil.make_archive(dst, "zip", RESULTS)
p = dst + ".zip"
print(f"zipped {p} ({os.path.getsize(p)/1e3:.0f} KB)")
try:
    from google.colab import files; files.download(p)
    print("download started")
except Exception:
    print(f"grab it from the Files pane: {p}")

---

### What this notebook contributes to the submission

1. **Real hyperspectral imagery over the actual study area.** Not a spectral library,
   not an analogue city — a 60 m, 230-band EMIT scene over Mussafah acquired in summer
   at 15:04 local time.
2. **A measurement multispectral imagery cannot make.** There is no Sentinel-2 band at
   1730 nm. The C–H enrichment that separates bituminous surfaces from merely dark ones
   is only available from a spectrometer.
3. **A quantified independence test.** ARI ≈ 0.09 against brightness-only clustering
   says the composition partition carries information albedo does not — which is the
   whole case for flying a spectrometer over a city.
4. **An honest negative.** ~67% of the scene is spectrally average. Aeolian dust
   homogenises surface spectra in the Gulf, and any 813-mission hyperspectral product
   over an arid city will meet the same ceiling. Better that the hackathon hears it
   from a team that measured it than discovers it in year two.

Back to [`02_main_analysis.ipynb`](02_main_analysis.ipynb) for the thermal sensitivity
and the roof ranking that these material classes attach to.